In [1]:
import pandas as pd
import numpy as np

In [2]:
car=pd.read_csv('Dataset_car.csv')

In [3]:
car.head()

,name,company,year,Price,kms_driven,fuel_type
0,Hyundai Santro Xing XO eRLX Euro III,Hyundai,2007,"80,000","45,000 kms",Petrol
1,Mahindra Jeep CL550 MDI,Mahindra,2006,"4,25,000",40 kms,Diesel
2,Maruti Suzuki Alto 800 Vxi,Maruti,2018,Ask For Price,"22,000 kms",Petrol
3,Hyundai Grand i10 Magna 1.2 Kappa VTVT,Hyundai,2014,"3,25,000","28,000 kms",Petrol
4,Ford EcoSport Titanium 1.5L TDCi,Ford,2014,"5,75,000","36,000 kms",Diesel


In [4]:
car.shape # size of the dataset 

(892, 6)

In [5]:
 car.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 892 entries, 0 to 891
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   name        892 non-null    object
 1   company     892 non-null    object
 2   year        892 non-null    object
 3   Price       892 non-null    object
 4   kms_driven  840 non-null    object
 5   fuel_type   837 non-null    object
dtypes: object(6)
memory usage: 41.9+ KB


## Quality of the Data_set 

1. year as non-year values
2. year object to int
3. price has objects so we need to convert it into int
4. kms_driven has km with int
5. kms_driven, fuel_type has nan value.
6. keep first 3 words of name

## CLEANING 

In [6]:
backup=car.copy() #creating a backup if something goes wrong 

In [7]:
car=car[car['year'].str.isnumeric()] #keeping only the cars with year as numeric value 

In [8]:
car['year']=car['year'].astype(int) # converting object file to int 

In [9]:
car=car[car['Price']!="Ask For Price"] # removing the Ask For Price

In [10]:
car['Price']=car['Price'].str.replace(",",'').astype(int)

In [11]:
car['kms_driven']=car['kms_driven'].str.split(' ').str.get(0).str.replace(",","")

In [12]:
car=car[car['kms_driven'].str.isnumeric()]

In [13]:
car['kms_driven']=car['kms_driven'].astype(int)

In [14]:
car=car[~car['fuel_type'].isna()]

In [15]:
car['name']=car['name'].str.split(" ").str.slice(0,3).str.join(" ")

In [16]:
car=car.reset_index(drop=True)

In [17]:
car=car[car['Price']<6e6].reset_index(drop=True)

In [18]:
car

,name,company,year,Price,kms_driven,fuel_type
0,Hyundai Santro Xing,Hyundai,2007,80000,45000,Petrol
1,Mahindra Jeep CL550,Mahindra,2006,425000,40,Diesel
2,Hyundai Grand i10,Hyundai,2014,325000,28000,Petrol
3,Ford EcoSport Titanium,Ford,2014,575000,36000,Diesel
4,Ford Figo,Ford,2012,175000,41000,Diesel
...,...,...,...,...,...,...
810,Maruti Suzuki Ritz,Maruti,2011,270000,50000,Petrol
811,Tata Indica V2,Tata,2009,110000,30000,Diesel
812,Toyota Corolla Altis,Toyota,2009,300000,132000,Petrol
813,Tata Zest XM,Tata,2018,260000,27000,Diesel


In [19]:
car.to_csv('cleaned_car_csv')

## Model

In [20]:
x=car.drop(columns='Price')
y=car['Price']

In [21]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2)

In [22]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline 

In [23]:
ohe=OneHotEncoder()
ohe.fit(x[['name','company','fuel_type']])

,categories,'auto'
,drop,None
,sparse_output,True
,dtype,<class 'numpy.float64'>
,handle_unknown,'error'
,min_frequency,None
,max_categories,None
,feature_name_combiner,'concat'


In [24]:
ohe.categories_

[array(['Audi A3 Cabriolet', 'Audi A4 1.8', 'Audi A4 2.0', 'Audi A6 2.0',
        'Audi A8', 'Audi Q3 2.0', 'Audi Q5 2.0', 'Audi Q7', 'BMW 3 Series',
        'BMW 5 Series', 'BMW 7 Series', 'BMW X1', 'BMW X1 sDrive20d',
        'BMW X1 xDrive20d', 'Chevrolet Beat', 'Chevrolet Beat Diesel',
        'Chevrolet Beat LS', 'Chevrolet Beat LT', 'Chevrolet Beat PS',
        'Chevrolet Cruze LTZ', 'Chevrolet Enjoy', 'Chevrolet Enjoy 1.4',
        'Chevrolet Sail 1.2', 'Chevrolet Sail UVA', 'Chevrolet Spark',
        'Chevrolet Spark 1.0', 'Chevrolet Spark LS', 'Chevrolet Spark LT',
        'Chevrolet Tavera LS', 'Chevrolet Tavera Neo', 'Datsun GO T',
        'Datsun Go Plus', 'Datsun Redi GO', 'Fiat Linea Emotion',
        'Fiat Petra ELX', 'Fiat Punto Emotion', 'Force Motors Force',
        'Force Motors One', 'Ford EcoSport', 'Ford EcoSport Ambiente',
        'Ford EcoSport Titanium', 'Ford EcoSport Trend',
        'Ford Endeavor 4x4', 'Ford Fiesta', 'Ford Fiesta SXi', 'Ford Figo',
        '

In [25]:
column_trans= make_column_transformer((OneHotEncoder(categories=ohe.categories_),['name','company','fuel_type']),
                                      remainder="passthrough")

In [26]:
lr=LinearRegression()

In [27]:
pipe= make_pipeline(column_trans,lr)

In [28]:
pipe.fit(x_train,y_train)

,steps,"[('columntransformer', ...), ('linearregression', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('onehotencoder', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [29]:
y_pred=pipe.predict(x_test)

In [30]:
r2_score(y_test,y_pred)

0.7223038632457285

In [37]:
score=[]
for i in range(1000):
    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2,random_state=i)
    lr=LinearRegression()
    pipe= make_pipeline(column_trans,lr)
    pipe.fit(x_train,y_train)
    y_pred=pipe.predict(x_test)
    score.append(r2_score(y_test,y_pred))

In [38]:
np.argmax(score)

np.int64(433)

In [39]:
score[np.argmax(score)]

0.8456957793726847

In [41]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2,random_state=np.argmax(score))
lr=LinearRegression()
pipe= make_pipeline(column_trans,lr)
pipe.fit(x_train,y_train)
y_pred=pipe.predict(x_test)
r2_score(y_test,y_pred)

0.8456957793726847

In [42]:
import pickle  

In [43]:
pickle.dump(pipe,open('Linear_Regression_Model.pkl','wb'))

In [62]:
pipe.predict(
    pd.DataFrame(
        [['Maruti Suzuki Swift', 'Maruti', 2019, 100, 'Petrol']],
        columns=['name', 'company', 'year', 'kms_driven', 'fuel_type']
    )
)

array([458934.66740154])